In [1]:
from pathlib import Path
import json
from ultralytics import YOLO
import random
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple
import time
import shutil
import cv2

from annotation_methods.budget_splits import make_nested_splits, Params
from annotation_methods.yolo_helpers import convert_all_inst_splits_to_yolo, write_yolo_dataset_yaml, train_yolo_model, run_inference, yolo_pred_txt_to_coco_results


In [2]:
# ---- Reusable constants ----
DATA_ROOT = Path("../../Data")
DATA_YAML_NAME = "dataset.yaml"
STATS_JSON_NAME = "stats.json"
RUNS_DIR = Path("runs")
MODEL_WEIGHTS = "yolo11m.pt"
INSTANCE_BUDGETS = (250, 500, 1000)
NUM_REPEATS = 3

RESULTS_PATH = Path("../../Results/Experiment_1")

DATASETS = ["apples", "tomatoes"]
DATASET_DICT = {
    "apples": ["good apple", "bad apple"],
    "tomatoes": ["tomato"],
}

## make the splits and yolo yaml file

In [3]:
for dataset in DATASETS:
    make_nested_splits(
        train_json=DATA_ROOT / dataset / "annotations" / "instances_train.json",
        out_root=DATA_ROOT / dataset / "yolo_splits",
        params=Params(INSTANCE_BUDGETS, val_frac=0.2, num_repeats=NUM_REPEATS, seed=42),
    )

Wrote splits to: /home/warredv/Thesis_WDV/Data/apples/yolo_splits
Wrote splits to: /home/warredv/Thesis_WDV/Data/tomatoes/yolo_splits


In [5]:
for dataset in DATASETS:
    all_results = convert_all_inst_splits_to_yolo(
        inst_values=INSTANCE_BUDGETS,
        num_repeats=NUM_REPEATS,
        coco_json=DATA_ROOT / dataset / "annotations" / "instances_train.json",
        dataset_root=DATA_ROOT / dataset,
        splits_root=DATA_ROOT / dataset / "yolo_splits",
    )
    for r in all_results:
        print(f"{dataset} inst{r['inst']} TRAIN:", r["train"])
        print(f"{dataset} inst{r['inst']} VAL  :", r["val"])

apples inst250 TRAIN: {'processed_images': 16, 'labels_written': 208, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst250 VAL  : {'processed_images': 3, 'labels_written': 52, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst250 TRAIN: {'processed_images': 14, 'labels_written': 196, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst250 VAL  : {'processed_images': 4, 'labels_written': 59, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst250 TRAIN: {'processed_images': 15, 'labels_written': 202, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst250 VAL  : {'processed_images': 4, 'labels_writt

In [7]:
for dataset in DATASETS:
    for inst in INSTANCE_BUDGETS:
        for repeat in range(NUM_REPEATS):
            split_dir = DATA_ROOT / dataset / "yolo_splits" / f"inst{inst}_r{repeat}"
            write_yolo_dataset_yaml(split_dir)

## train models

In [4]:
# train all models
for dataset in DATASET_DICT.keys():
    for inst_budget in INSTANCE_BUDGETS:
        for repeat in range(NUM_REPEATS):
            result = train_yolo_model(
                dataset=dataset,
                split=f"yolo_splits/inst{inst_budget}_r{repeat}",
                model_weights=MODEL_WEIGHTS,
                data_root=DATA_ROOT,
                yaml_name=DATA_YAML_NAME,
                runs_dir=RUNS_DIR,
            )
            print(result["model_name"], result["machine_training_time_s"], result["num_initial_bbox"])

New https://pypi.org/project/ultralytics/8.3.245 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.11.14 torch-2.9.1 CUDA:0 (NVIDIA A10G, 22588MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../Data/apples/yolo_splits/inst250_r0/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11m_inst250_

## Run Inference

In [9]:
# run inference on all models
for dataset in DATASET_DICT.keys():
    for inst_budget in INSTANCE_BUDGETS:
        for repeat in range(NUM_REPEATS):
            run_inference(
                dataset=dataset, 
                model_name=f"yolo11m_inst{inst_budget}_r{repeat}",
                data_root=DATA_ROOT,
                runs_dir=RUNS_DIR,
                outdir_base=Path("yolo_outputs"),
                warmup_required=True
            )

Results saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst250_r0
31 labels saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst250_r0/labels
Results saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst250_r1
31 labels saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst250_r1/labels
Results saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst250_r2
31 labels saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst250_r2/labels
Results saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst500_r0
31 labels saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst500_r0/labels
Results saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_yolo11m_inst500_r1
31 labels saved to /home/warredv/Thesis_WDV/Models/Yolo/yolo_outputs/apples_test_y

## Convert to COCO format

In [10]:
# run inference on all models
for dataset in DATASET_DICT.keys():
    for inst_budget in INSTANCE_BUDGETS:
        for repeat in range(NUM_REPEATS):
            yolo_pred_txt_to_coco_results(
                dataset=dataset,
                model_name=f"yolo11m_inst{inst_budget}_r{repeat}",
                categories_list=DATASET_DICT.get(dataset),
                data_root=DATA_ROOT,
                outdir_base=Path("yolo_outputs"),
                results_path=RESULTS_PATH, 
                is_xywh_normalized=True,
                has_conf=True,
            )